# 🎨 Онлайн-доска — запуск в Google Colab

Лаунчер: **клонирует репозиторий → ставит зависимости → запускает сервер и публикует ссылку через ngrok.**

**Перед запуском:**
1. В ячейке 1 укажите `REPO_URL` — адрес *вашего форка* репозитория.
2. Добавьте свой ngrok-токен в секреты Colab (слева 🔑 **Secrets** → новый секрет `NGROK_AUTHTOKEN`, включите *Notebook access*). Токен можно получить на <https://dashboard.ngrok.com/get-started/your-authtoken>. Если секрет не задан — ячейка 3 спросит токен скрытым вводом.

Затем **Runtime → Run all**. В конце появится публичная ссылка вида `https://xxxx.ngrok-free.app`.


In [ ]:
# 1) Клонирование репозитория (замените URL на свой форк)
REPO_URL    = "https://github.com/USERNAME/o-canvas.git"   # ← ваш форк
PROJECT_DIR = "o-canvas"

import os, subprocess

if not os.path.isdir(PROJECT_DIR):
    print("Клонирую репозиторий…")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, PROJECT_DIR], check=True)
else:
    print("Репозиторий уже есть — обновляю (git pull)…")
    subprocess.run(["git", "-C", PROJECT_DIR, "pull", "--ff-only"], check=False)

os.chdir(PROJECT_DIR)
print("Рабочая папка:", os.getcwd())
print("Файлы:", ", ".join(sorted(os.listdir())))


In [ ]:
# 2) Установка зависимостей
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Зависимости установлены.")


In [ ]:
# 3) Токен ngrok + запуск сервера и туннеля
import os, sys, time, threading, subprocess, urllib.request, getpass

PORT = 8000

# --- Токен ngrok БЕЗ хардкода: секреты Colab -> переменная окружения -> скрытый ввод ---
def get_ngrok_token():
    try:
        from google.colab import userdata          # секреты Colab (рекомендуется)
        t = userdata.get("NGROK_AUTHTOKEN")
        if t:
            print("Токен ngrok получен из секретов Colab.")
            return t.strip()
    except Exception:
        pass
    t = os.environ.get("NGROK_AUTHTOKEN")           # переменная окружения
    if t:
        print("Токен ngrok получен из переменной окружения.")
        return t.strip()
    return getpass.getpass("Вставьте ваш ngrok authtoken: ").strip()  # ручной ввод

NGROK_TOKEN = get_ngrok_token()
assert NGROK_TOKEN, (
    "Токен ngrok не задан. Добавьте секрет NGROK_AUTHTOKEN в Colab (🔑 Secrets) "
    "или введите токен вручную. "
    "Получить: https://dashboard.ngrok.com/get-started/your-authtoken")

# --- Зависимость туннеля ---
try:
    from pyngrok import ngrok, exception
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok"], check=True)
    from pyngrok import ngrok, exception

ngrok.set_auth_token(NGROK_TOKEN)
os.makedirs("boards", exist_ok=True)

# --- FastAPI поднимаем один раз в фоне; ошибки старта перехватываем ---
_server_error = {}
if not globals().get("_WB_STARTED"):
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())

    def run_uvicorn():
        try:
            import uvicorn
            uvicorn.run("app:app", host="0.0.0.0", port=PORT, log_level="warning")
        except Exception as e:                       # импорт app.py упал и т.п.
            _server_error["err"] = e
            import traceback; traceback.print_exc()

    threading.Thread(target=run_uvicorn, daemon=True).start()
    globals()["_WB_STARTED"] = True

# --- Активная проверка готовности (по IPv4 127.0.0.1 — тот же адрес отдаём ngrok) ---
def wait_until_ready(url="http://127.0.0.1:%d/" % PORT, timeout=40):
    start = time.time()
    while time.time() - start < timeout:
        if _server_error.get("err"):
            raise RuntimeError("FastAPI не запустился: %r" % _server_error["err"])
        try:
            with urllib.request.urlopen(url, timeout=2) as r:
                if r.status == 200:
                    return True
        except Exception:
            time.sleep(0.5)
    return False

if not wait_until_ready():
    raise RuntimeError(
        "Сервер FastAPI не ответил на http://127.0.0.1:%d за 40 c.\n"
        "→ Проверьте, что ячейки 1–2 выполнены, затем Runtime → Restart session." % PORT)
print("✅ Локальный сервер отвечает на http://127.0.0.1:%d" % PORT)

# --- ngrok: закрыть старые туннели и открыть новый ---
# ВАЖНО: адрес "127.0.0.1:PORT" (IPv4), а НЕ число PORT — иначе агент резолвит localhost в ::1,
# где сервер не слушает, и возвращает ERR_NGROK_8012.
def open_tunnel(addr="127.0.0.1:%d" % PORT, tries=5):
    try:
        for t in ngrok.get_tunnels():
            ngrok.disconnect(t.public_url)
    except Exception:
        pass
    ngrok.kill()
    for i in range(tries):
        time.sleep(3 if i == 0 else 5)
        try:
            return ngrok.connect(addr, "http").public_url
        except exception.PyngrokNgrokHTTPError as e:
            if "ERR_NGROK_334" in str(e) or "already online" in str(e):
                print(f"  прежний туннель ещё закрывается, попытка {i + 1}…")
                ngrok.kill()
                continue
            raise
    raise RuntimeError(
        "ngrok: прежний туннель не освободился.\n"
        "→ Runtime → Restart session, либо остановите старый туннель на "
        "https://dashboard.ngrok.com/endpoints (на бесплатном плане доступен один туннель).")

public_url = open_tunnel()
print("\n✅ Доска запущена. Ссылка для совместной работы:")
print("   ", public_url)
print("    Отдельная комната/доска — добавьте хэш в конец, например:", public_url + "#room1")
print("\nПри первом открытии нажмите на странице ngrok кнопку «Visit Site».")
